# Whatdog

In [ ]:
import os
import torch.utils.data as data
import torchvision.datasets as datasets
import torchvision.transforms as transforms
import torchvision.models as models
import torch
import torch.nn as nn
import numpy as np
import random
from torch.utils.data import DataLoader, Dataset

import matplotlib.pyplot as plt

In [ ]:
def set_seed(seed=24):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

In [ ]:
set_seed(24)

## Data

In [ ]:

current_dir = os.getcwd()
root_dir = current_dir + "/Images"

In [ ]:
image_transformation = transforms.Compose([
    transforms.Resize(size=(256, 256)),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    # Normalize based on Imagenet1k dataset
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
dog_dataset = datasets.ImageFolder(root=root_dir, transform=image_transformation)

In [ ]:
generator=torch.Generator().manual_seed(42)
train_size = int(0.8 * len(dog_dataset))
test_size = len(dog_dataset) - train_size
train_dataset, test_dataset = data.random_split(dataset=dog_dataset, lengths=[train_size, test_size], generator=generator)

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64)

In [ ]:
train_features, train_labels = next(iter(train_loader))
print(f"Feature batch shape: {train_features.size()}")
print(f"Labels batch shape: {train_labels.size()}")
img = train_features[0].squeeze()
label = train_labels[0]
img_data_transposed = img.permute(1, 2, 0).cpu().numpy()
plt.imshow(img_data_transposed)
plt.show()
print(f"Label: {label}")

## Model

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print(device)

In [ ]:
res50_ft = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
num_classes = 120

In [ ]:
for param in res50_ft.parameters():
    param.requires_grad = False

In [ ]:
res50_ft.fc = nn.Linear(res50_ft.fc.in_features, num_classes)

In [ ]:
print(res50_ft)

In [ ]:
# Load the pretrained Resnet18 model
res18_ft = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

# Freeze the model parameters
for param in res18_ft.parameters():
    param.requires_grad = False

# Update the final classification layer (updatable parameters)
num_classes = 120
res18_ft.fc = nn.Linear(res18_ft.fc.in_features, num_classes)

In [ ]:
loss_fn = torch.nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, res18_ft.parameters()),
    lr=1e-3,
    weight_decay=0.01
)

In [ ]:
def training_loop(model, train_loader, loss_fn, optimizer, num_epochs=5, device=device):
    """
    Trains a model using the provided arguments and hyperparameters.
    Arguments:
    model: A pytorch model (nn.Module)
    train_loader: A pytorch DataLoader that is loaded with the training data (Dataloader)
    loss_fn: A loss function used to calculate the loss
    optimizer: An optimizer for updating the weights of the model
    """
    model.to(device)
    model.train()
    running_loss = 0
    num_batches = 0

    for epoch in range(num_epochs):
        running_loss = 0
        num_batches = 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            output = model(images)
            loss = loss_fn(output, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss
            num_batches += 1
        print(f"EPOCH {epoch} loss: {running_loss / num_batches}")


 

In [ ]:
training_loop(
    model=res18_ft,
    train_loader=train_loader,
    loss_fn=loss_fn,
    optimizer=optimizer,
    device=device
)

In [ ]:
def evaluate_model(model, test_loader, device=device):
    """
        Evaluates a pytorch classification model
        Args:
            model (nn.Module): The model that will be evaluated.
            test_loader (DataLoader): A pytorch dataloader with the test data.
            device (torch.device): The device that the tensors will be run on.

        Returns:
            The percentage of correct classificaitons (float)

    """
    model.to(device)
    model.eval()
    with torch.no_grad():
        
        
        correct = 0
        total = 0
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)

            _, classification = outputs.max(1)

            total += labels.size(0)
            correct += classification.eq(labels).sum().item()
    print(f"{correct=} \n{total=}")
    return 100. * correct / total

In [ ]:
evaluate_model(
    model=res18_ft, 
    test_loader=test_loader,
    device=device
)

### Model Evaluation Results:  

correct=3060  
total=4116

74.34402332361516% Accuracy